# Build a Tiny Event Loop, Then Compare It With asyncio

Companion notebook for the [Event Loops and Reactors](https://codemia.io/courses/concurrency_and_async_programming/cc_event_loops_and_reactors) and [Async/Await Under the Hood](https://codemia.io/courses/concurrency_and_async_programming/cc_async_await_under_the_hood) lessons on Codemia.

**What you will build**: a working event loop in about 40 lines, using plain Python generators as coroutines. It has a ready queue, a timer heap and a virtual clock. You will run the same program on your loop and on `asyncio` and check that both print the same order, then watch one blocking call stall every other task, and measure that stall as loop lag.

## Setup

In [ ]:
%pip install -q matplotlib

In [ ]:
import asyncio
import heapq
import time
from collections import deque

import matplotlib.pyplot as plt

## 1. Coroutines are paused functions

A generator stops at every `yield` and hands a value to whoever is driving it. That is all a coroutine needs. Our tasks will yield small requests to the loop: `("sleep", seconds)` to wait for a timer, or `None` to give other tasks a turn.

In [ ]:
def sleep(seconds):
    yield ("sleep", seconds)

def pause():
    yield None

def worker(name, delay, log):
    log.append(f"{name} start")
    yield from sleep(delay)
    log.append(f"{name} woke after {delay}")
    yield from pause()
    log.append(f"{name} done")

## 2. The loop: a ready queue, a timer heap and a clock

Each iteration runs every task that is ready right now, one step each. A task that asks to sleep goes onto the timer heap, ordered by wake up time and then by a sequence number so ties stay first come, first served. When nothing is ready, the loop jumps the clock straight to the next timer. That jump is what makes the loop **virtual time**: ten thousand sleeping tasks cost nothing while they sleep.

In [ ]:
class TinyLoop:
    def __init__(self):
        self.now = 0.0
        self.ready = deque()
        self.timers = []
        self.seq = 0
        self.steps = 0

    def spawn(self, coro):
        self.ready.append(coro)

    def run(self):
        while self.ready or self.timers:
            if not self.ready:
                self.now = max(self.now, self.timers[0][0])
            while self.timers and self.timers[0][0] <= self.now:
                _, _, coro = heapq.heappop(self.timers)
                self.ready.append(coro)
            for _ in range(len(self.ready)):  # only the tasks ready at the start of this pass
                coro = self.ready.popleft()
                self.steps += 1
                try:
                    req = next(coro)
                except StopIteration:
                    continue
                if req is None:
                    self.ready.append(coro)
                else:
                    self.seq += 1
                    heapq.heappush(self.timers, (self.now + req[1], self.seq, coro))

loop = TinyLoop()
log = []
for name, d in [("a", 2), ("b", 1), ("c", 2)]:
    loop.spawn(worker(name, d, log))
loop.run()
print("virtual time", loop.now, "steps", loop.steps)
log

## 3. Run the same program on asyncio

`asyncio` keeps the same two structures, a ready deque and a timer heap (`loop._ready` and `loop._scheduled` in CPython). If our loop is faithful, the log order should match. We scale the delays down to 10 ms so the real run is quick.

Notebooks already run inside an event loop, so we `await` directly instead of calling `asyncio.run`.

In [ ]:
async def aworker(name, delay, log):
    log.append(f"{name} start")
    await asyncio.sleep(delay * 0.01)
    log.append(f"{name} woke after {delay}")
    await asyncio.sleep(0)
    log.append(f"{name} done")

alog = []
await asyncio.gather(*(aworker(n, d, alog) for n, d in [("a", 2), ("b", 1), ("c", 2)]))
print("same order:", alog == log)
alog

## 4. One blocking call stalls everyone

The loop can only switch tasks at a `yield` or `await`. A task that does CPU work or calls a blocking function holds the only thread, and every timer that comes due in the meantime fires late. We measure this with a **lag monitor**: a task that sleeps 20 ms over and over and records how late it actually woke up. Partway through, a handler calls `time.sleep(0.15)`, which never yields.

In [ ]:
async def lag_monitor(samples, period=0.02, count=40):
    for _ in range(count):
        t0 = time.perf_counter()
        await asyncio.sleep(period)
        samples.append((time.perf_counter() - t0 - period) * 1000)

async def bad_handler():
    await asyncio.sleep(0.3)
    time.sleep(0.15)          # blocking: no await, the loop is frozen

async def good_handler():
    await asyncio.sleep(0.3)
    await asyncio.to_thread(time.sleep, 0.15)  # same wait, moved off the loop thread

runs = {}
for name, handler in [("blocking call", bad_handler), ("to_thread", good_handler)]:
    samples = []
    await asyncio.gather(lag_monitor(samples), handler())
    runs[name] = samples
    print(f"{name:14s} max lag {max(samples):6.1f} ms")

In [ ]:
plt.figure(figsize=(7, 3.2))
for name, s in runs.items():
    plt.plot(s, marker=".", label=name)
plt.xlabel("monitor wake up #")
plt.ylabel("lag (ms)")
plt.title("A blocking call shows up as one large lag spike")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Why one thread can hold ten thousand waits

Waiting tasks cost a heap entry, not a thread. Spawn 10,000 tasks that each sleep a random amount on your loop and count the steps. With threads you would need 10,000 stacks; here the loop does about two steps per task and the virtual clock simply jumps between timers.

In [ ]:
import random
random.seed(7)

def napper(log):
    yield from sleep(random.randint(1, 1000))
    log.append(1)

big = TinyLoop()
done = []
for _ in range(10_000):
    big.spawn(napper(done))
t0 = time.perf_counter()
big.run()
print(f"{len(done)} tasks finished, {big.steps} steps, virtual time {big.now:.0f}, wall {time.perf_counter() - t0:.3f}s")

## 6. Exercises

1. Add a `("wait", task)` request so one coroutine can wait for another to finish, then rebuild `asyncio.gather` on top of it.
2. Give the loop an I/O poller: a request `("read", sock)` that parks the task until `selectors.DefaultSelector` reports the socket readable. Serve a few connections with it.
3. Change the timer heap to break ties by name instead of by sequence number and find a program whose log order changes. Why does `asyncio` use a sequence number?
4. Move the CPU work in `bad_handler` into `loop.run_in_executor` with a `ProcessPoolExecutor` and compare the lag with `to_thread`. When does the GIL make the thread version insufficient?